# 5. Geographic Map
### Austin Airbnb Pricing Intelligence — Notebook 5 of 5
---

## 5.1 Business Context

Every previous notebook communicated insights through tables and charts.
Notebook 5 communicates through **geography**.

An interactive map lets stakeholders see the complete Austin market at once —
10,402 listings plotted, color-coded by price tier, and clickable for details.
This is the "wow" deliverable that brings the entire analysis to life.

## 5.2 What This Notebook Produces

A standalone HTML file — `austin_listings_map.html` — that anyone can open
in their browser to explore the Austin market visually.

Four layers of information:
- All 10,402 listings as circle markers
- Color coding by price tier
- Click popups with full listing details
- Heat map overlay showing price density

## 5.3 Technology

**Folium** — Python wrapper for Leaflet.js.
Same mapping engine used by Uber, Airbnb, Yelp.
Exports to self-contained HTML with no server required.

---
## 5.4 Step 1 — Price Tier Segmentation

Before plotting, listings are segmented into four price tiers.
These tiers become the color legend for the map.

| Tier | Price Range | Color |
|---|---|---|
| Budget | Under $150 | Green |
| Mid | $150 – $250 | Yellow |
| Upper-mid | $250 – $400 | Orange |
| Premium | $400+ | Red |

The thresholds come from the quartile analysis in Chart 1 (Notebook 3).

In [19]:
# ─── Notebook 5 — Geographic Map Setup ───────────────────────────────────────
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster, FastMarkerCluster
import os

# Load clean dataset
listings = pd.read_csv('../data/processed/listings_clean.csv')

# Output path
output_dir = '../outputs/'
os.makedirs(output_dir, exist_ok=True)

print("NOTEBOOK 5 SETUP COMPLETE")
print("=" * 50)
print(f"  Dataset loaded   : {listings.shape[0]:,} rows")
print(f"  Listings with    : lat/lng coordinates")
print(f"  Valid coords     : {listings[['latitude', 'longitude']].dropna().shape[0]:,}")
print(f"  Output directory : {output_dir}")

NOTEBOOK 5 SETUP COMPLETE
  Dataset loaded   : 10,402 rows
  Listings with    : lat/lng coordinates
  Valid coords     : 10,402
  Output directory : ../outputs/


---
## 5.4 Step 1 — Price Tier Segmentation

Before plotting, listings are segmented into four price tiers.
These tiers become the color legend for the map.

| Tier | Price Range | Color |
|---|---|---|
| Budget | Under $150 | Green |
| Mid | $150 – $250 | Yellow |
| Upper-mid | $250 – $400 | Orange |
| Premium | $400+ | Red |

The thresholds come from the quartile analysis in Chart 1 (Notebook 3).

In [20]:
# ─── Step 1 — Price Tier Segmentation ────────────────────────────────────────

def price_tier(price):
    if price < 150:
        return 'Budget'
    elif price < 250:
        return 'Mid'
    elif price < 400:
        return 'Upper-mid'
    else:
        return 'Premium'

listings['price_tier'] = listings['price'].apply(price_tier)

tier_colors = {
    'Budget'    : '#00703C',
    'Mid'       : '#F4B400',
    'Upper-mid' : '#FF6F00',
    'Premium'   : '#C00000',
}

tier_counts = listings['price_tier'].value_counts()
print("PRICE TIER DISTRIBUTION")
print("=" * 50)
for tier in ['Budget', 'Mid', 'Upper-mid', 'Premium']:
    count = tier_counts.get(tier, 0)
    pct = count / len(listings) * 100
    bar = '█' * int(pct / 2)
    print(f"  {tier:10s} : {count:>5,} listings ({pct:>5.1f}%)  {bar}")

PRICE TIER DISTRIBUTION
  Budget     : 5,850 listings ( 56.2%)  ████████████████████████████
  Mid        : 2,324 listings ( 22.3%)  ███████████
  Upper-mid  : 1,140 listings ( 11.0%)  █████
  Premium    : 1,088 listings ( 10.5%)  █████


---
## 5.5 Step 2 — Building the Base Map

The map is centered on Austin's geographic center with zoom level 11 —
wide enough to see all neighborhoods, close enough to distinguish listings.

**Tile layer:** OpenStreetMap — free, detailed, and professional looking.

In [21]:
# ─── Step 2 — Build Base Map with Satellite + Labels ─────────────────────────

# Austin center coordinates
austin_center = [30.2672, -97.7431]

# Create the map with satellite imagery
m = folium.Map(
    location      = austin_center,
    zoom_start    = 11,
    tiles         = None,  # No default tiles
    control_scale = True
)

# Layer 1: Satellite imagery from Esri
folium.TileLayer(
    tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr  = 'Esri World Imagery',
    name  = 'Satellite',
    overlay = False,
    control = True
).add_to(m)

# Layer 2: Street labels overlay (CartoDB)
folium.TileLayer(
    tiles = 'https://cartodb-basemaps-{s}.global.ssl.fastly.net/rastertiles/voyager_only_labels/{z}/{x}/{y}.png',
    attr  = 'CartoDB',
    name  = 'Street Labels',
    overlay = True,
    control = True
).add_to(m)

# Alternative view: Clean light map
folium.TileLayer(
    tiles = 'CartoDB positron',
    attr  = 'CartoDB',
    name  = 'Light Map',
    overlay = False,
    control = True
).add_to(m)

print("BASE MAP CREATED")
print(f"  Center      : Austin, TX ({austin_center})")
print(f"  Zoom level  : 11")
print(f"  Tile layers : Satellite + Labels + Light Map (toggle-able)")

BASE MAP CREATED
  Center      : Austin, TX ([30.2672, -97.7431])
  Zoom level  : 11
  Tile layers : Satellite + Labels + Light Map (toggle-able)


---
## 5.6 Step 3 — Adding All 10,402 Listings

Each listing becomes a circle marker on the map.
Color reflects the price tier.
Click any marker to see full listing details.

**Performance note:** At 10,402 markers, Folium uses `FastMarkerCluster`
to group nearby markers at low zoom levels — keeping the map responsive.

In [22]:
# ─── Step 3 — Add Circle Markers for All Listings ────────────────────────────

# Drop listings without coordinates (shouldn't happen after cleaning)
geo_data = listings.dropna(subset=['latitude', 'longitude']).copy()

# Create feature groups for each tier (toggle-able via layer control)
tier_layers = {}
for tier, color in tier_colors.items():
    tier_layers[tier] = folium.FeatureGroup(
        name = f'{tier} (<${150 if tier=="Budget" else 250 if tier=="Mid" else 400 if tier=="Upper-mid" else "400+"})',
        show = True
    )

# Add each listing as a circle marker to its tier layer
for _, row in geo_data.iterrows():
    tier  = row['price_tier']
    color = tier_colors[tier]

    popup_html = f"""
    <div style="font-family: sans-serif; width: 250px;">
        <h4 style="margin: 0 0 8px 0; color: {color};">${row['price']:.0f}/night</h4>
        <b>Room Type:</b> {row['room_type']}<br>
        <b>Bedrooms:</b> {row['bedrooms']:.0f}<br>
        <b>Bathrooms:</b> {row['bathrooms']:.1f}<br>
        <b>Accommodates:</b> {row['accommodates']}<br>
        <b>Neighborhood:</b> {row['neighbourhood_cleansed']}<br>
        <b>Superhost:</b> {'Yes' if row['host_is_superhost'] else 'No'}<br>
        <b>Occupied Nights:</b> {row['estimated_occupancy_l365d']:.0f}/yr<br>
        <b>Annual Revenue:</b> ${row['estimated_revenue_l365d']:,.0f}<br>
        <b>Rating:</b> {row['review_scores_rating']:.2f}
    </div>
    """

    folium.CircleMarker(
        location    = [row['latitude'], row['longitude']],
        radius      = 3,
        color       = color,
        fill        = True,
        fill_color  = color,
        fill_opacity= 0.7,
        weight      = 0.5,
        popup       = folium.Popup(popup_html, max_width=300)
    ).add_to(tier_layers[tier])

# Add all tier layers to the map
for tier in ['Budget', 'Mid', 'Upper-mid', 'Premium']:
    tier_layers[tier].add_to(m)

print(f"  Markers added : {len(geo_data):,} across 4 tier layers")

  Markers added : 10,402 across 4 tier layers


---
## 5.7 Step 4 — Heat Map Overlay

A heat map overlay shows where premium listings concentrate geographically.
Hot spots indicate neighborhoods with high density of expensive listings.

The heat map is a separate toggle-able layer — users can turn it on or off.

In [23]:
# ─── Step 4 — Add Heat Map Overlay ────────────────────────────────────────────

# Use price as intensity — premium listings create hot spots
heat_data = [
    [row['latitude'], row['longitude'], row['price']]
    for _, row in geo_data.iterrows()
]

heat_layer = folium.FeatureGroup(name='Price Heat Map', show=False)
HeatMap(
    heat_data,
    radius   = 12,
    blur     = 15,
    max_zoom = 13,
    gradient = {
        0.2 : '#00703C',
        0.4 : '#F4B400',
        0.6 : '#FF6F00',
        0.8 : '#C00000',
        1.0 : '#800000'
    }
).add_to(heat_layer)
heat_layer.add_to(m)

print("  Heat map layer added (toggle-able)")

  Heat map layer added (toggle-able)


---
## 5.8 Step 5 — Legend and Layer Control

The map needs a visual legend so users can decode the colors.
Layer control lets users toggle tiers on and off individually.

In [24]:
# ─── Step 5 — Legend and Layer Control ────────────────────────────────────────

# Add layer control (toggle tiers on/off)
folium.LayerControl(collapsed=False, position='topright').add_to(m)

# Custom HTML legend
legend_html = """
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background-color: white;
    padding: 15px 20px;
    border: 1px solid #999;
    border-radius: 6px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    font-family: sans-serif;
    font-size: 13px;
">
    <b style="font-size: 14px;">Austin Airbnb Price Tiers</b><br>
    <div style="margin-top: 10px;">
        <span style="background:#00703C; width:14px; height:14px; display:inline-block; margin-right:8px; border-radius:50%;"></span>
        Budget (&lt;$150)<br>
        <span style="background:#F4B400; width:14px; height:14px; display:inline-block; margin-right:8px; border-radius:50%;"></span>
        Mid ($150–$250)<br>
        <span style="background:#FF6F00; width:14px; height:14px; display:inline-block; margin-right:8px; border-radius:50%;"></span>
        Upper-mid ($250–$400)<br>
        <span style="background:#C00000; width:14px; height:14px; display:inline-block; margin-right:8px; border-radius:50%;"></span>
        Premium ($400+)
    </div>
    <div style="margin-top: 10px; font-size: 11px; color: #666;">
        10,402 listings · Inside Airbnb · 2026
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Title banner
title_html = """
<div style="
    position: fixed;
    top: 12px;
    left: 50%;
    transform: translateX(-50%);
    z-index: 9999;
    background-color: white;
    padding: 10px 20px;
    border: 1px solid #999;
    border-radius: 6px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.2);
    font-family: sans-serif;
">
    <b style="font-size: 15px;">Austin Airbnb Pricing Intelligence</b>
    <span style="color: #666; font-size: 12px; margin-left: 10px;">
        Click any listing for details · Toggle layers on the right
    </span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

print("  Legend added (bottom-left)")
print("  Layer control added (top-right)")
print("  Title banner added (top-center)")

  Legend added (bottom-left)
  Layer control added (top-right)
  Title banner added (top-center)


---
## 5.9 Step 6 — Export to HTML

The final map is saved as a standalone HTML file.
Any browser can open it — no server, no installation required.

The file is self-contained — it includes all the map data,
the base tiles, and the interactive JavaScript.

In [25]:
# ─── Step 6 — Export Map to HTML ──────────────────────────────────────────────

output_path = f'{output_dir}austin_listings_map.html'
m.save(output_path)

file_size_mb = os.path.getsize(output_path) / 1024 / 1024

print("MAP EXPORTED")
print("=" * 50)
print(f"  File     : {output_path}")
print(f"  Size     : {file_size_mb:.2f} MB")
print(f"  Status   : Ready to open in any browser")
print(f"\n  Open the file directly or drag it into a browser window.")

MAP EXPORTED
  File     : ../outputs/austin_listings_map.html
  Size     : 15.28 MB
  Status   : Ready to open in any browser

  Open the file directly or drag it into a browser window.


---
## 5.10 Notebook Summary

The final deliverable is `austin_listings_map.html` — an interactive map
of every Austin Airbnb listing in the dataset.

### What the Map Shows

10,402 listings plotted across the Austin metro area,
each colored by price tier. Clicking any marker reveals the listing's
price, bedroom count, superhost status, occupancy, and annual revenue.

A toggleable heat map overlay highlights where premium listings cluster.
Three base layers — satellite, light map, and street labels —
can be switched to suit the viewer's preference.

### What the Geography Reveals

Central Austin — 78701, 78703, 78704 — shows dense red markers.
This is the downtown and South Congress premium zone.

The lake districts to the west — 78730, 78734, 78746 — show
scattered red markers on larger lots. This is the luxury waterfront market
where prices exceed $500/night.

North and northeast Austin — 78753, 78758, 78723 — appear predominantly green.
This is the accessible entry-market zone where budget listings concentrate.

78702 in East Austin shows every color represented on the map.
It is the most heterogeneous neighborhood in the city —
and also the one with the highest listing volume.

### How to Use the Map

Anyone with the repo can open `outputs/austin_listings_map.html`
directly in a browser. No server, no installation, no setup.
The file is self-contained.

For a recruiter reviewing the portfolio, this is the chart that makes
the entire analysis tangible. Numbers become places.

### Project Status

All five notebooks are complete. The pricing intelligence project
now consists of:

- A clean analytical dataset
- 10 SQL queries covering market, neighborhoods, amenities, hosts, and recommendations
- 12 IBCS-compliant charts telling the complete story
- A linear regression pricing model with honest limitation disclosure
- An interactive geographic map

Next steps if time permits: Tableau dashboard for executive summary,
and a final README polish pointing to the live HTML map.